In [16]:
!pip install dagshub mlflow -q

In [17]:
import os
import gc
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import dagshub
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("XGBoost")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())
print("✅ XGBoost version:", xgb.__version__)

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow
✅ XGBoost version: 3.2.0


# **Cleaning**

In [18]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD   = 0.80
    ID_MISSING_THRESHOLD    = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    test_id.columns  = [c.replace('-', '_') for c in test_id.columns]
    train_id.columns = [c.replace('-', '_') for c in train_id.columns]

    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id,  on="TransactionID", how="left")

    y_train_xgb = train["isFraud"].copy()
    X_train_xgb = train.drop(columns=["isFraud", "TransactionID"])
    X_test_xgb  = test.drop(columns=["TransactionID"])

    del train, test, train_txn, train_id, test_txn, test_id
    gc.collect()

    id_like_cols  = [c for c in X_train_xgb.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train_xgb.columns if c not in id_like_cols]

    missing_ratio = X_train_xgb.isnull().mean()
    drop_txn      = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id       = [c for c in id_like_cols  if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing  = drop_txn + drop_id

    X_train_xgb = X_train_xgb.drop(columns=drop_missing)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in drop_missing if c in X_test_xgb.columns])

    near_constant_cols = []
    for col in X_train_xgb.columns:
        top_freq = X_train_xgb[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train_xgb = X_train_xgb.drop(columns=near_constant_cols)
    X_test_xgb  = X_test_xgb.drop(columns=[c for c in near_constant_cols if c in X_test_xgb.columns])

    for col in X_train_xgb.columns:
        if col not in X_test_xgb.columns:
            X_test_xgb[col] = np.nan
    X_test_xgb = X_test_xgb[X_train_xgb.columns]

    mlflow.log_param("txn_missing_threshold",   TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold",    ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)
    mlflow.log_metric("train_rows",             int(X_train_xgb.shape[0]))
    mlflow.log_metric("test_rows",              int(X_test_xgb.shape[0]))
    mlflow.log_metric("features_after_cleaning",int(X_train_xgb.shape[1]))
    mlflow.log_metric("dropped_high_missing",   len(drop_missing))
    mlflow.log_metric("dropped_near_constant",  len(near_constant_cols))
    mlflow.log_metric("fraud_rate",             float(y_train_xgb.mean()))

    print(f"X_train: {X_train_xgb.shape}")
    print(f"X_test:  {X_test_xgb.shape}")
    print(f"Fraud rate: {y_train_xgb.mean():.4f}")

    X_train_clean_xgb = X_train_xgb
    X_test_clean_xgb  = X_test_xgb
    y_train_clean_xgb = y_train_xgb

X_train: (590540, 353)
X_test:  (506691, 353)
Fraud rate: 0.0350
🏃 View run XGBoost_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/42725f65445a4f2490016a568e7f0431
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Engineering

In [19]:
with mlflow.start_run(run_name="XGBoost_FeatureEngineering"):
    X_train = X_train_clean_xgb.copy()
    X_test  = X_test_clean_xgb.copy()
    y_train = y_train_clean_xgb.copy()

    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"]  = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"]  = np.sin(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)
    X_test["hour_cos"]  = np.cos(2 * np.pi * ((X_test["TransactionDT"]  // 3600) % 24) / 24)

    X_train["day_of_week"] = (X_train["TransactionDT"] // 86400) % 7
    X_test["day_of_week"]  = (X_test["TransactionDT"]  // 86400) % 7

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test  = X_test.drop(columns=["TransactionDT"],  errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    for c in cat_cols:
        mapping = {v: i for i, v in enumerate(X_train[c].astype(str).unique())}
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c]  = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)
    
    group_cols = ["card1", "card2", "card3", "addr1", "P_emaildomain", "R_emaildomain"]
    agg_features_added = 0

    for col in group_cols:
        if col not in X_train.columns:
            continue

        fraud_mean = y_train.groupby(X_train[col]).mean()
        X_train[f"{col}_fraud_rate"]      = X_train[col].map(fraud_mean).fillna(y_train.mean())
        X_test[f"{col}_fraud_rate"]       = X_test[col].map(fraud_mean).fillna(y_train.mean())

        amt_mean = X_train.groupby(col)["TransactionAmt"].mean()
        X_train[f"{col}_amt_mean"]        = X_train[col].map(amt_mean).fillna(X_train["TransactionAmt"].mean())
        X_test[f"{col}_amt_mean"]         = X_test[col].map(amt_mean).fillna(X_train["TransactionAmt"].mean())

        freq = X_train[col].value_counts()
        X_train[f"{col}_freq"]            = X_train[col].map(freq).fillna(0)
        X_test[f"{col}_freq"]             = X_test[col].map(freq).fillna(0)

        agg_features_added += 3

    if "card1_amt_mean" in X_train.columns:
        X_train["amt_vs_card1_mean"] = X_train["TransactionAmt"] - X_train["card1_amt_mean"]
        X_test["amt_vs_card1_mean"]  = X_test["TransactionAmt"]  - X_test["card1_amt_mean"]
        agg_features_added += 1

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("cat_encoding",          "ordinal_train_mapping")
    mlflow.log_param("amt_log_transform",     True)
    mlflow.log_param("cyclic_time_encoding",  True)
    mlflow.log_param("day_of_week",           True)
    mlflow.log_param("aggregation_features",  True)
    mlflow.log_param("agg_group_cols",        str(group_cols))
    mlflow.log_metric("features_after_fe",    int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded",     len(cat_cols))
    mlflow.log_metric("agg_features_added",   agg_features_added)

    print(f"X_train_fe: {X_train.shape}")
    print(f"X_test_fe:  {X_test.shape}")
    print(f"Aggregation features added: {agg_features_added}")

    X_train_fe_xgb = X_train
    X_test_fe_xgb  = X_test
    y_train_fe_xgb = y_train

X_train_fe: (590540, 375)
X_test_fe:  (506691, 375)
Aggregation features added: 19
🏃 View run XGBoost_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/b76ebe10be1b4317ba3d6706655a18e0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Feature Selection

In [20]:
with mlflow.start_run(run_name="XGBoost_FeatureSelection"):
    X_train = X_train_fe_xgb.copy()
    X_test  = X_test_fe_xgb.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test  = X_test.replace([np.inf, -np.inf],  np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test  = X_test.drop(columns=const_cols,  errors="ignore")

    CORR_THRESHOLD = 0.98
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    sample_n = min(120_000, len(X_train))
    idx  = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()
    upper     = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > CORR_THRESHOLD).any()]

    protected_cols = [c for c in drop_corr if any(
        kw in c for kw in ["_fraud_rate", "_amt_mean", "_freq", "amt_vs_"]
    )]
    drop_corr = [c for c in drop_corr if c not in protected_cols]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test  = X_test.drop(columns=drop_corr,  errors="ignore")
    X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_param("corr_threshold",          CORR_THRESHOLD)
    mlflow.log_param("protected_agg_features",  len(protected_cols))
    mlflow.log_metric("dropped_const",           len(const_cols))
    mlflow.log_metric("dropped_high_corr",       len(drop_corr))
    mlflow.log_metric("features_after_fs",       int(X_train.shape[1]))

    print(f"X_train_fs: {X_train.shape}")
    print(f"X_test_fs:  {X_test.shape}")
    print(f"Protected aggregation features: {len(protected_cols)}")

    X_train_final_xgb = X_train
    X_test_final_xgb  = X_test

X_train_fs: (590540, 321)
X_test_fs:  (506691, 321)
Protected aggregation features: 0
🏃 View run XGBoost_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/9fcafc7ee5ea4379993aa718e66ca0d8
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


# Training

In [21]:
import pickle
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import numpy as np
import pandas as pd
import mlflow

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X_train_final_xgb, y_train_fe_xgb,
    test_size=0.10, random_state=42, stratify=y_train_fe_xgb
)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.20, random_state=42, stratify=y_dev
)

neg = int((y_tr == 0).sum())
pos = int((y_tr == 1).sum())
spw = round(neg / pos, 2)

print(f"Train size:    {X_tr.shape}")
print(f"Val size:      {X_val.shape}")
print(f"Holdout size:  {X_holdout.shape}")
print(f"scale_pos_weight = {spw}  (neg/pos = {neg}/{pos})")

Train size:    (425188, 321)
Val size:      (106298, 321)
Holdout size:  (59054, 321)
scale_pos_weight = 27.58  (neg/pos = 410310/14878)


In [22]:
with mlflow.start_run(run_name="XGB_Baseline"):
    mlflow.log_params({
        "n_estimators": 500, "max_depth": 6, "learning_rate": 0.1,
        "scale_pos_weight": spw, "tree_method": "hist", "device": "cuda",
        "note": "baseline with class balancing + early stopping"
    })

    clf = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        scale_pos_weight=spw, tree_method="hist", device="cuda",
        eval_metric="auc", early_stopping_rounds=50, random_state=42
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    best_iter = clf.best_iteration
    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr,  iteration_range=(0, best_iter+1))[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val, iteration_range=(0, best_iter+1))[:, 1])
    gap = train_auc - val_auc

    mlflow.log_metrics({
        "train_auc": round(train_auc, 5),
        "val_auc":   round(val_auc,   5),
        "overfit_gap": round(gap, 5),
        "best_iteration": best_iter
    })
    print(f"[Baseline] Best Iter: {best_iter} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")


[Baseline] Best Iter: 499 | Train: 0.9902 | Val: 0.9670 | Gap: 0.0232
🏃 View run XGB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/1a65a360dceb44438b3ad02397597623
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [23]:
for n_est in [500, 1000, 2000, 3000]:
    with mlflow.start_run(run_name=f"XGB_nestimators_{n_est}"):
        clf = xgb.XGBClassifier(
            n_estimators=n_est, max_depth=6, learning_rate=0.1,
            scale_pos_weight=spw, tree_method="hist", device="cuda",
            eval_metric="auc", early_stopping_rounds=50, random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        best_iter  = clf.best_iteration
        train_auc  = roc_auc_score(y_tr,  clf.predict_proba(X_tr,  iteration_range=(0, best_iter+1))[:, 1])
        val_auc    = roc_auc_score(y_val, clf.predict_proba(X_val, iteration_range=(0, best_iter+1))[:, 1])
        gap = train_auc - val_auc

        mlflow.log_params({"n_estimators": n_est, "max_depth": 6, "learning_rate": 0.1,
                           "scale_pos_weight": spw, "tree_method": "hist", "device": "cuda"})
        mlflow.log_metrics({"train_auc": round(train_auc, 5), "val_auc": round(val_auc, 5),
                            "overfit_gap": round(gap, 5), "best_iteration": best_iter})

        converged = "CONVERGED" if best_iter < n_est - 1 else "STILL LEARNING — increase n_est"
        print(f"  n_est={n_est:<5} | Best Iter: {best_iter:<4} | Val: {val_auc:.4f} | Gap: {gap:.4f} | {converged}")


  n_est=500   | Best Iter: 499  | Val: 0.9670 | Gap: 0.0232 | STILL LEARNING — increase n_est
🏃 View run XGB_nestimators_500 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/81488517617b4cdf846e2ea9b5dc97ec
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_est=1000  | Best Iter: 994  | Val: 0.9712 | Gap: 0.0260 | CONVERGED
🏃 View run XGB_nestimators_1000 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/30c4fdd5f9f34f8594aa15c189cb19be
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_est=2000  | Best Iter: 1571 | Val: 0.9723 | Gap: 0.0269 | CONVERGED
🏃 View run XGB_nestimators_2000 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/5b3bf1e0df1d4bedb50d28fbb9705295
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  n_est

In [24]:
depth_results = []
for depth in [4, 5, 6, 7, 8]:
    for mcw in [10, 50, 100, 200]:
        run_label = f"depth{depth}_mcw{mcw}"
        with mlflow.start_run(run_name=f"XGB_{run_label}"):
            clf = xgb.XGBClassifier(
                n_estimators=2000,          
                max_depth=depth,
                min_child_weight=mcw,
                learning_rate=0.05,         
                scale_pos_weight=spw,
                tree_method="hist", device="cuda",
                eval_metric="auc",
                early_stopping_rounds=100,
                random_state=42
            )
            clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            best_iter  = clf.best_iteration
            train_auc  = roc_auc_score(y_tr,  clf.predict_proba(X_tr,  iteration_range=(0, best_iter+1))[:, 1])
            val_auc    = roc_auc_score(y_val, clf.predict_proba(X_val, iteration_range=(0, best_iter+1))[:, 1])
            gap = train_auc - val_auc

            mlflow.log_params({"max_depth": depth, "min_child_weight": mcw,
                               "n_estimators_ceiling": 2000, "learning_rate": 0.05})
            mlflow.log_metrics({"train_auc": round(train_auc, 5), "val_auc": round(val_auc, 5),
                                "overfit_gap": round(gap, 5), "best_iteration": best_iter})

            depth_results.append({
                "max_depth": depth, "min_child_weight": mcw,
                "train_auc": train_auc, "val_auc": val_auc,
                "gap": gap, "best_iter": best_iter
            })
            print(f"  {run_label:<20} | Best Iter: {best_iter:<4} | "
                  f"Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

depth_df = pd.DataFrame(depth_results)

depth_df["converged"] = depth_df["best_iter"] < 1999
depth_df["stability"] = depth_df["val_auc"] - 3 * (depth_df["gap"] - 0.020).clip(lower=0)

pool = depth_df[depth_df["converged"]] if depth_df["converged"].any() else depth_df
best_depth_row = pool.loc[pool["stability"].idxmax()]
best_depth = int(best_depth_row["max_depth"])
best_mcw   = int(best_depth_row["min_child_weight"])

print(f"\n{'run':<22} | {'val_auc':>8} | {'gap':>6} | {'stability':>10} | converged")
print("-" * 65)
for _, row in pool.sort_values("stability", ascending=False).head(10).iterrows():
    print(f"  depth{int(row['max_depth'])}_mcw{int(row['min_child_weight']):<13} | "
          f"{row['val_auc']:>8.4f} | {row['gap']:>6.4f} | {row['stability']:>10.4f} | "
          f"{'yes' if row['converged'] else 'no'}")

print(f"\n-> Best: depth={best_depth}, min_child_weight={best_mcw}")
print(f"   Val AUC={best_depth_row['val_auc']:.4f} | Gap={best_depth_row['gap']:.4f}")

if best_depth_row["train_auc"] > 0.999:
    print("WARNING: train_auc still ~1.0. Consider increasing min_child_weight to 300+.")


  depth4_mcw10         | Best Iter: 1999 | Train: 0.9815 | Val: 0.9628 | Gap: 0.0186
🏃 View run XGB_depth4_mcw10 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/b621467eead04beb85943ca538d08a7c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth4_mcw50         | Best Iter: 1999 | Train: 0.9816 | Val: 0.9630 | Gap: 0.0185
🏃 View run XGB_depth4_mcw50 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/acb9e82a848f40759c6862fc572d8c14
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  depth4_mcw100        | Best Iter: 1999 | Train: 0.9810 | Val: 0.9630 | Gap: 0.0179
🏃 View run XGB_depth4_mcw100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/41423b15575b41869bcde242a5b6d983
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [25]:
sampling_results = []
for subsample, colsample in [
    (1.0, 1.0),
    (0.9, 0.9),
    (0.8, 0.8),
    (0.8, 0.6),
    (0.7, 0.7),
    (0.7, 0.5),
    (0.6, 0.6),
]:
    label = f"sub{subsample}_col{colsample}"
    with mlflow.start_run(run_name=f"XGB_sampling_{label}"):
        clf = xgb.XGBClassifier(
            n_estimators=2000,
            max_depth=best_depth,
            min_child_weight=best_mcw,
            learning_rate=0.05,
            subsample=subsample,
            colsample_bytree=colsample,
            scale_pos_weight=spw,
            tree_method="hist", device="cuda",
            eval_metric="auc",
            early_stopping_rounds=100,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        best_iter  = clf.best_iteration
        train_auc  = roc_auc_score(y_tr,  clf.predict_proba(X_tr,  iteration_range=(0, best_iter+1))[:, 1])
        val_auc    = roc_auc_score(y_val, clf.predict_proba(X_val, iteration_range=(0, best_iter+1))[:, 1])
        gap = train_auc - val_auc

        mlflow.log_params({"subsample": subsample, "colsample_bytree": colsample,
                           "n_estimators_ceiling": 2000})
        mlflow.log_metrics({"train_auc": round(train_auc, 5), "val_auc": round(val_auc, 5),
                            "gap": round(gap, 5), "best_iteration": best_iter})

        sampling_results.append({
            "label": label, "sub": subsample, "col": colsample,
            "train_auc": train_auc, "val_auc": val_auc,
            "gap": gap, "best_iter": best_iter
        })
        converged = "CONVERGED" if best_iter < 1999 else "STILL LEARNING"
        print(f"  {label:<22} | Best Iter: {best_iter:<4} | Val: {val_auc:.4f} | Gap: {gap:.4f} | {converged}")

samp_df = pd.DataFrame(sampling_results)
samp_df["converged"] = samp_df["best_iter"] < 1999
samp_df["stability"] = samp_df["val_auc"] - 3 * (samp_df["gap"] - 0.020).clip(lower=0)

pool_s = samp_df[samp_df["converged"]] if samp_df["converged"].any() else samp_df
best_samp_row  = pool_s.loc[pool_s["stability"].idxmax()]
best_subsample = float(best_samp_row["sub"])
best_colsample = float(best_samp_row["col"])

print(f"\n-> Best sampling: subsample={best_subsample}, colsample={best_colsample}")
print(f"   Val AUC={best_samp_row['val_auc']:.4f} | Gap={best_samp_row['gap']:.4f}")

  sub1.0_col1.0          | Best Iter: 1997 | Val: 0.9670 | Gap: 0.0208 | CONVERGED
🏃 View run XGB_sampling_sub1.0_col1.0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/f8ed7ba699fa455980dae496aa988e31
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.9_col0.9          | Best Iter: 1999 | Val: 0.9686 | Gap: 0.0212 | STILL LEARNING
🏃 View run XGB_sampling_sub0.9_col0.9 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/c911cf8ecac04dd1b00d9cbe08cd1463
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  sub0.8_col0.8          | Best Iter: 1999 | Val: 0.9688 | Gap: 0.0212 | STILL LEARNING
🏃 View run XGB_sampling_sub0.8_col0.8 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/42b79f631a9f4fc5b7601a71130c9d29
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-

In [26]:
reg_results = []
for alpha, lam, gamma in [
    (0,   1,  0),
    (1,   1,  0),
    (0,   5,  0),
    (1,   5,  0),
    (1,   5,  1),
    (2,   5,  1),
    (1,  10,  1),
    (2,  10,  2),
]:
    label = f"a{alpha}_l{lam}_g{gamma}"
    with mlflow.start_run(run_name=f"XGB_reg_{label}"):
        clf = xgb.XGBClassifier(
            n_estimators=2000,
            max_depth=best_depth,
            min_child_weight=best_mcw,
            learning_rate=0.05,
            subsample=best_subsample,
            colsample_bytree=best_colsample,
            reg_alpha=alpha, reg_lambda=lam, gamma=gamma,
            scale_pos_weight=spw,
            tree_method="hist", device="cuda",
            eval_metric="auc",
            early_stopping_rounds=100,
            random_state=42
        )
        clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        best_iter  = clf.best_iteration
        train_auc  = roc_auc_score(y_tr,  clf.predict_proba(X_tr,  iteration_range=(0, best_iter+1))[:, 1])
        val_auc    = roc_auc_score(y_val, clf.predict_proba(X_val, iteration_range=(0, best_iter+1))[:, 1])
        gap = train_auc - val_auc

        mlflow.log_params({"reg_alpha": alpha, "reg_lambda": lam, "gamma": gamma,
                           "n_estimators_ceiling": 2000})
        mlflow.log_metrics({"train_auc": round(train_auc, 5), "val_auc": round(val_auc, 5),
                            "gap": round(gap, 5), "best_iteration": best_iter})

        reg_results.append({"label": label, "alpha": alpha, "lambda": lam, "gamma": gamma,
                            "train_auc": train_auc, "val_auc": val_auc,
                            "gap": gap, "best_iter": best_iter})
        converged = "CONVERGED" if best_iter < 1999 else "STILL LEARNING"
        print(f"  {label:<15} | Best Iter: {best_iter:<4} | Val: {val_auc:.4f} | Gap: {gap:.4f} | {converged}")

reg_df = pd.DataFrame(reg_results)
reg_df["converged"] = reg_df["best_iter"] < 1999
reg_df["stability"] = reg_df["val_auc"] - 3 * (reg_df["gap"] - 0.020).clip(lower=0)

pool_r = reg_df[reg_df["converged"]] if reg_df["converged"].any() else reg_df
best_reg_row = pool_r.loc[pool_r["stability"].idxmax()]
best_alpha   = float(best_reg_row["alpha"])
best_lambda  = float(best_reg_row["lambda"])
best_gamma   = float(best_reg_row["gamma"])

print(f"\n-> Best reg: alpha={best_alpha}, lambda={best_lambda}, gamma={best_gamma}")
print(f"   Val AUC={best_reg_row['val_auc']:.4f} | Gap={best_reg_row['gap']:.4f}")

  a0_l1_g0        | Best Iter: 1997 | Val: 0.9670 | Gap: 0.0208 | CONVERGED
🏃 View run XGB_reg_a0_l1_g0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/29f2d2e94db3401aad9eaf6f5015c75c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  a1_l1_g0        | Best Iter: 1999 | Val: 0.9674 | Gap: 0.0209 | STILL LEARNING
🏃 View run XGB_reg_a1_l1_g0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/520142b30be14197a7e3c5980c8334cd
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  a0_l5_g0        | Best Iter: 1999 | Val: 0.9668 | Gap: 0.0209 | STILL LEARNING
🏃 View run XGB_reg_a0_l5_g0 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/03dfc89a22dc40eb9109717841c1328d
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
  a1_l5_g0        

In [27]:
with mlflow.start_run(run_name="XGB_CrossValidation_5fold"):
    final_params = {
        "max_depth":        int(best_depth),
        "min_child_weight": int(best_mcw),
        "learning_rate":    0.05,
        "subsample":        float(best_subsample),
        "colsample_bytree": float(best_colsample),
        "reg_alpha":        float(best_alpha),
        "reg_lambda":       float(best_lambda),
        "gamma":            float(best_gamma),
        "scale_pos_weight": float(spw),
        "tree_method":      "hist",
        "device":           "cuda",
        "random_state":     42
    }

    mlflow.log_params(final_params)
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("cv_data", "X_dev (90% of train — holdout excluded)")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_val_aucs   = []
    cv_train_aucs = []
    cv_iters      = []

    print(f"Running 5-fold CV on {len(X_dev):,} dev rows (holdout untouched)")

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_dev, y_dev)):
        X_f_tr, X_f_val = X_dev.iloc[tr_idx], X_dev.iloc[val_idx]
        y_f_tr, y_f_val = y_dev.iloc[tr_idx], y_dev.iloc[val_idx]

        model = xgb.XGBClassifier(
            n_estimators=5000,
            **final_params,
            eval_metric="auc",
            early_stopping_rounds=100
        )
        model.fit(X_f_tr, y_f_tr, eval_set=[(X_f_val, y_f_val)], verbose=False)

        best_f_iter = model.best_iteration
        cv_iters.append(best_f_iter)

        val_probs   = model.predict_proba(X_f_val, iteration_range=(0, best_f_iter+1))[:, 1]
        train_probs = model.predict_proba(X_f_tr,  iteration_range=(0, best_f_iter+1))[:, 1]

        val_auc_f   = roc_auc_score(y_f_val, val_probs)
        train_auc_f = roc_auc_score(y_f_tr,  train_probs)

        cv_val_aucs.append(val_auc_f)
        cv_train_aucs.append(train_auc_f)

        mlflow.log_metric("fold_val_auc",   round(val_auc_f,   5), step=fold)
        mlflow.log_metric("fold_train_auc", round(train_auc_f, 5), step=fold)
        mlflow.log_metric("fold_best_iter", best_f_iter,           step=fold)

        print(f"  Fold {fold+1}: Best Iter={best_f_iter:<4} | Train={train_auc_f:.4f} | Val={val_auc_f:.4f} | Gap={train_auc_f-val_auc_f:.4f}")

    cv_mean = np.mean(cv_val_aucs)
    cv_std  = np.std(cv_val_aucs)
    cv_gap  = np.mean(cv_train_aucs) - cv_mean
    final_n_est = int(np.mean(cv_iters)) + 50

    mlflow.log_metrics({
        "cv_mean_val_auc":  round(cv_mean,   5),
        "cv_std_val_auc":   round(cv_std,    5),
        "cv_mean_gap":      round(cv_gap,    5),
        "cv_avg_best_iter": int(np.mean(cv_iters)),
        "final_n_estimators": final_n_est
    })

    print(f"\nCV Mean Val AUC: {cv_mean:.4f} (+/- {cv_std:.4f})")
    print(f"CV Mean Gap:     {cv_gap:.4f}")
    print(f"-> Final n_estimators set to: {final_n_est}")

Running 5-fold CV on 531,486 dev rows (holdout untouched)
  Fold 1: Best Iter=4998 | Train=0.9963 | Val=0.9723 | Gap=0.0240
  Fold 2: Best Iter=4995 | Train=0.9964 | Val=0.9714 | Gap=0.0250
  Fold 3: Best Iter=4997 | Train=0.9966 | Val=0.9737 | Gap=0.0229
  Fold 4: Best Iter=4978 | Train=0.9965 | Val=0.9715 | Gap=0.0251
  Fold 5: Best Iter=4996 | Train=0.9962 | Val=0.9732 | Gap=0.0230

CV Mean Val AUC: 0.9724 (+/- 0.0009)
CV Mean Gap:     0.0240
-> Final n_estimators set to: 5042
🏃 View run XGB_CrossValidation_5fold at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/cebb5075338e417d8a6adaff177dc3e0
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4


In [29]:

with mlflow.start_run(run_name="XGB_Final_Model") as run:
    neg_dev = int((y_dev == 0).sum())
    pos_dev = int((y_dev == 1).sum())
    spw_dev = round(neg_dev / pos_dev, 2)

    final_params_full = {
        "n_estimators":      final_n_est,
        "max_depth":         int(best_depth),
        "min_child_weight":  int(best_mcw),
        "learning_rate":     0.05,
        "subsample":         float(best_subsample),
        "colsample_bytree":  float(best_colsample),
        "reg_alpha":         float(best_alpha),
        "reg_lambda":        float(best_lambda),
        "gamma":             float(best_gamma),
        "scale_pos_weight":  spw_dev,
        "tree_method":       "hist",
        "device":            "cuda",
        "random_state":      42
    }

    mlflow.log_params(final_params_full)
    mlflow.log_param("trained_on",      "X_dev (full, 90% of train data)")
    mlflow.log_param("evaluated_on",    "X_holdout (10% never seen in tuning)")
    mlflow.log_param("n_est_source",    "CV avg best_iter + 50 buffer")
    mlflow.log_param("cv_mean_val_auc", round(cv_mean, 5))

    final_clf = xgb.XGBClassifier(
        **final_params_full,
        eval_metric="auc"
    )
    final_clf.fit(X_dev, y_dev, verbose=100)

    holdout_probs = final_clf.predict_proba(X_holdout)[:, 1]
    dev_probs     = final_clf.predict_proba(X_dev)[:, 1]

    holdout_auc = roc_auc_score(y_holdout, holdout_probs)
    dev_auc     = roc_auc_score(y_dev,     dev_probs)
    gap = dev_auc - holdout_auc

    mlflow.log_metrics({
        "dev_auc":     round(dev_auc,     5),
        "holdout_auc": round(holdout_auc, 5),
        "overfit_gap": round(gap,         5)
    })

    if dev_auc > 0.999:
        print(f"WARNING: dev_auc={dev_auc:.4f} ~1.0 — model still overfitting. Increase min_child_weight or gamma.")
    if gap > 0.035:
        print(f"WARNING: Gap={gap:.4f} > 0.035 — consider more regularisation.")
    else:
        print(f"OK: Gap={gap:.4f}")

    print(f"\nFinal Model Results:")
    print(f"  Dev AUC:     {dev_auc:.4f}")
    print(f"  Holdout AUC: {holdout_auc:.4f}  <- honest estimate")
    print(f"  Gap:         {gap:.4f}")
    print(f"  n_estimators used: {final_n_est}")
    print(f"  Params: LR=0.05, Depth={best_depth}, MCW={best_mcw}, "
          f"sub={best_subsample}, col={best_colsample}, "
          f"alpha={best_alpha}, lambda={best_lambda}, gamma={best_gamma}")

    mlflow.xgboost.log_model(
        xgb_model=final_clf,
        artifact_path="xgboost_final",
        registered_model_name="XGBoost_FraudDetection"
    )

    import pickle
    pkl_path = "xgboost_final_model.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(final_clf, f)
    mlflow.log_artifact(pkl_path)

    print(f"\nRun ID: {run.info.run_id}")

    test_probs = final_clf.predict_proba(X_test_final_xgb)[:, 1]
    submission = pd.DataFrame({
        "TransactionID": pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv")["TransactionID"],
        "isFraud": test_probs
    })
    submission.to_csv("submission_xgboost.csv", index=False)
    mlflow.log_artifact("submission_xgboost.csv")
    print(f"Submission saved: {submission.shape}")
    print(submission.head())

2026/05/05 15:46:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


OK: Gap=0.0224

Final Model Results:
  Dev AUC:     0.9960
  Holdout AUC: 0.9736  <- honest estimate
  Gap:         0.0224
  n_estimators used: 5042
  Params: LR=0.05, Depth=5, MCW=200, sub=1.0, col=1.0, alpha=0.0, lambda=1.0, gamma=0.0


Registered model 'XGBoost_FraudDetection' already exists. Creating a new version of this model...
2026/05/05 15:46:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_FraudDetection, version 7
Created version '7' of model 'XGBoost_FraudDetection'.



Run ID: 527682f46a3c43faa2b79b7469c67d3c
Submission saved: (506691, 2)
   TransactionID       isFraud
0        3663549  6.961286e-05
1        3663550  6.633835e-03
2        3663551  4.525272e-07
3        3663552  2.878399e-03
4        3663553  1.702606e-03
🏃 View run XGB_Final_Model at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4/runs/527682f46a3c43faa2b79b7469c67d3c
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/4
